In [1]:
# Parameters
UF = "AC"


## Seção 3.6.2 - Representatividade temporal 

Este notebook reproduz a figura 29 da seção 3.6.2 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Figura 29 - Representatividade temporal das medições nas estações de monitoramento da qualidade do ar.

A figura apresenta a representatividade temporal calculada a partir da proporção de dados válidos em cada intervalo de agregação, considerando critérios mínimos de completude definidos para cada nível temporal. 

Ao clicar na estação, é possível verificar o nome do município, ID_MMA da estação, ID_OEMA, e os anos monitorados e o percentual de dados válidos. Os anos considerados representativos estão destacados em verde.

In [2]:
from pathlib import Path
from IPython.display import IFrame
import requests

# Mapeamento sigla -> código IBGE (necessário para a API de malhas)
UF_CODES = {
    "AC": 12, "AL": 27, "AP": 16, "AM": 13, "BA": 29, "CE": 23, "DF": 53,
    "ES": 32, "GO": 52, "MA": 21, "MT": 51, "MS": 50, "MG": 31, "PA": 15,
    "PB": 25, "PR": 41, "PE": 26, "PI": 22, "RJ": 33, "RN": 24, "RS": 43,
    "RO": 11, "RR": 14, "SC": 42, "SP": 35, "SE": 28, "TO": 17,
}

def build_rep_temporal_map(UF=None, output_path="temp_rep_temporal_map.html"):
    root = "https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/_static/representatividade/rep_temporal/"

    gj_diario = requests.get(root + "rep_temporal_diario.geojson").text
    gj_mensal = requests.get(root + "rep_temporal_mensal.geojson").text
    gj_anual  = requests.get(root + "rep_temporal_anual.geojson").text

    # -------------------------------------------------------------------
    # Fronteira do estado (opcional, apenas se UF for informado)
    # -------------------------------------------------------------------
    gj_boundary = "null"
    if UF:
        uf_code = UF_CODES.get(UF.upper())
        if uf_code is None:
            raise ValueError(f"UF '{UF}' não reconhecida.")
        boundary_url = (
            f"https://servicodados.ibge.gov.br/api/v2/malhas/{uf_code}"
            f"?resolucao=2&formato=application/vnd.geo+json"
        )
        resp = requests.get(boundary_url)
        if resp.ok:
            gj_boundary = resp.text
        else:
            gj_boundary = "null"  # segue sem a fronteira se a API falhar

    uf_js = f'"{UF.upper()}"' if UF else "null"

    # -------------------------------------------------------------------
    # GERAÇÃO DO HTML
    # -------------------------------------------------------------------
    html_code = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<style>
  #map {
      width:100%;
      height:640px;
      border:1px solid #ddd;
      position: relative;
  }
  #selector {
      margin: 10px 0;
      font-size: 14px;
      padding: 4px 6px;
  }
  .bar-container {
      display:flex;
      align-items:center;
      gap:6px;
      margin:2px 0;
  }
  .bar-bg {
      width:120px;
      height:10px;
      border:1px solid #aaa;
  }
  .bar-fill {
      height:100%;
  }
  .legend {
      position: absolute;
      bottom: 20px;
      right: 20px;
      background: white;
      padding: 10px;
      border-radius: 6px;
      font-size: 12px;
      line-height: 1.4;
      box-shadow: 0 1px 4px rgba(0,0,0,0.3);
      z-index: 9999;
  }
  .legend-title {
      font-weight: bold;
      margin-bottom: 6px;
  }
  .legend-item {
      display: flex;
      align-items: center;
      margin: 3px 0;
  }
  .legend-color {
      width: 14px;
      height: 14px;
      margin-right: 6px;
      display: inline-block;
      border: 1px solid #333;
  }
</style>
</head>

<body>

<select id="selector">
  <option value="diario">Representatividade diária</option>
  <option value="mensal">Representatividade mensal</option>
  <option value="anual">Representatividade anual</option>
</select>

<div id="map">
  <div class="legend" id="legend">
    <div class="legend-title">Representatividade (%)</div>
    <div class="legend-item"><span class="legend-color" style="background:#666;"></span>0%</div>
    <div class="legend-item"><span class="legend-color" style="background:#d73027;"></span>1–24%</div>
    <div class="legend-item"><span class="legend-color" style="background:#fc8d59;"></span>25–49%</div>
    <div class="legend-item"><span class="legend-color" style="background:#fee08b;"></span>50–74%</div>
    <div class="legend-item"><span class="legend-color" style="background:#d9ef8b;"></span>75–89%</div>
    <div class="legend-item"><span class="legend-color" style="background:#1a9850;"></span>90–100%</div>
  </div>
</div>

<script id="gj-diario" type="application/json">
""" + gj_diario + """
</script>

<script id="gj-mensal" type="application/json">
""" + gj_mensal + """
</script>

<script id="gj-anual" type="application/json">
""" + gj_anual + """
</script>

<script id="gj-boundary" type="application/json">
""" + gj_boundary + """
</script>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>

const UF_FILTER = """ + uf_js + """;

const map = L.map("map", {
    minZoom: 3.5,
    maxZoom: 20,
    maxBounds: L.latLngBounds([-34,-74],[6,-34]),
    maxBoundsViscosity: 0.8
}).setView([-14.2,-51.9], 4.5);

L.tileLayer(
    "https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_2ge2_1_675289d2b5268d90fee0fdab",
    {
        minZoom: 2,
        maxZoom: 20,
        maxNativeZoom: 20,
        subdomains: "abcd",
        attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
    }
).addTo(map);

function cor(v){
    if (v == null || isNaN(v)) return "#ffffff";
    if (v === 0) return "#666666";
    if (v < 25) return "#d73027";
    if (v < 50) return "#fc8d59";
    if (v < 75) return "#fee08b";
    if (v < 90) return "#d9ef8b";
    return "#1a9850";
}

function barra(val, col){
    if (val == null || isNaN(val)) return "—";
    return `
      <div class="bar-container">
        <div class="bar-bg">
          <div class="bar-fill" style="width:${val}%; background:${col};"></div>
        </div>
        <span>${val.toFixed(1)}%</span>
      </div>`;
}

function formatarAnos(monitorados, representativos) {
    if (!monitorados || monitorados.length === 0) return "—";
    const setRep = new Set(representativos || []);
    return monitorados.map(ano => {
        if (setRep.has(ano)) {
            return `<span style="color: #006400; font-weight: bold;">${ano}</span>`;
        }
        return ano;
    }).join(", ");
}

function popupHTML(p, campo){
    const v = p[campo];
    const anosFormatados = formatarAnos(p.ANOS_MONITORADOS, p.ANOS_REPRESENTATIVOS);
    return `
      <div style="font-size:13px;">
        <b>${p.CIDADE} (${p.UF})</b><br>
        <b>ID_MMA:</b> ${p.ID_MMA_COMPLETO}<br>
        <b>ID_OEMA:</b> ${p.ID_OEMA}<br>
        <div style="margin: 4px 0; line-height:1.4;">
            <b>Anos monitorados:</b><br>
            <span style="color:#555;">${anosFormatados}</span>
        </div>
        <hr style="margin:4px 0;">
        ${barra(v, cor(v))}
      </div>`;
}

const dataDiario = JSON.parse(document.getElementById("gj-diario").textContent);
const dataMensal = JSON.parse(document.getElementById("gj-mensal").textContent);
const dataAnual  = JSON.parse(document.getElementById("gj-anual").textContent);
const dataBoundaryRaw = document.getElementById("gj-boundary").textContent.trim();
const dataBoundary = (dataBoundaryRaw && dataBoundaryRaw !== "null") ? JSON.parse(dataBoundaryRaw) : null;

let layerAtual = null;
let boundaryLayer = null;

// Desenha a fronteira do estado (se houver) e ajusta o zoom
if (dataBoundary) {
    boundaryLayer = L.geoJSON(dataBoundary, {
        style: { color: "#000", weight: 2, fillOpacity: 0 }
    }).addTo(map);
    map.fitBounds(boundaryLayer.getBounds(), { padding: [20, 20] });
}

function carregar(dataset, campo){
    if (layerAtual) map.removeLayer(layerAtual);

    layerAtual = L.geoJSON(dataset, {
        filter: (f) => !UF_FILTER || f.properties.UF === UF_FILTER,
        pointToLayer: (f, latlng) => {
            const p = f.properties;
            const valor = p[campo];
            const c = cor(valor);

            const mk = L.circleMarker(latlng, {
                radius:6,
                color:c,
                weight:2,
                fillColor:c,
                fillOpacity:0.9
            });

            mk.bindPopup(popupHTML(p, campo));
            mk.bindTooltip(`${p.CIDADE} (${p.UF})`);
            return mk;
        }
    }).addTo(map);

    // Se não há fronteira desenhada mas há um filtro de UF, ajusta o zoom aos pontos
    if (!dataBoundary && UF_FILTER && layerAtual.getLayers().length > 0) {
        map.fitBounds(layerAtual.getBounds(), { padding: [30, 30] });
    }
}

// inicial
carregar(dataDiario, "PRCNT_REP_TEMPORAL_DIARIA");

document.getElementById("selector").addEventListener("change", (e) => {
    if (e.target.value === "diario") carregar(dataDiario, "PRCNT_REP_TEMPORAL_DIARIA");
    if (e.target.value === "mensal") carregar(dataMensal, "PRCNT_REP_TEMPORAL_MENSAL");
    if (e.target.value === "anual")  carregar(dataAnual,  "PRCNT_REP_TEMPORAL_ANUAL");
});
</script>

</body>
</html>
"""

    Path(output_path).write_text(html_code, encoding="utf-8")
    return html_code


# Uso:
# build_rep_temporal_map()          -> mapa do Brasil inteiro (como antes)
# build_rep_temporal_map(UF="SC")   -> mapa filtrado para Santa Catarina, com fronteira e zoom

In [3]:
html_code = build_rep_temporal_map(UF)

In [4]:
import webbrowser
import os

# Salvar figura interativa localmente e abrir no navegador
output_dir = "outputs"
output_path = os.path.join(output_dir, "Figura.29.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

webbrowser.open(output_path)

False